# assistente virtual dio

este notebook é uma versão para google colab do projeto de assistente virtual.

aqui o foco é testar text to speech com `gtts` e simular comandos por texto. o speech to text com microfone depende mais do ambiente local, então ficou no projeto python da pasta `src`.

## instalação das bibliotecas

vamos instalar as libs usadas no projeto.

In [ ]:
!pip install gTTS wikipedia SpeechRecognition

## text to speech com gtts

esta função recebe um texto, gera um mp3 e mostra o player de áudio no notebook.

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display


def speak(text, filename="output.mp3"):
    print(f"assistente: {text}")
    tts = gTTS(text=text, lang="pt-br")
    tts.save(filename)
    display(Audio(filename, autoplay=False))


speak("olá, eu sou um assistente virtual simples feito em python")

## comandos por texto

no colab, vamos simular os comandos digitando texto. isso evita depender de microfone.

In [ ]:
from datetime import datetime
from urllib.parse import quote_plus
import webbrowser

import wikipedia


wikipedia.set_lang("pt")

## função para responder comandos

a função abaixo interpreta comandos simples e devolve uma resposta.

In [ ]:
def limpar_termo_wikipedia(comando):
    texto = comando.lower().strip()
    for prefixo in ["pesquisar", "pesquise", "buscar", "busque", "wikipedia", "na wikipedia"]:
        texto = texto.replace(prefixo, " ")
    return " ".join(texto.split())


def responder_comando(comando):
    comando = comando.lower().strip()

    if not comando:
        return "não entendi. tenta escrever um comando."

    if "youtube" in comando:
        webbrowser.open("https://www.youtube.com")
        return "abrindo o youtube."

    if "horário" in comando or "horario" in comando or "que horas" in comando:
        agora = datetime.now().strftime("%H:%M")
        return f"agora são {agora}."

    if "farmácia" in comando or "farmacia" in comando:
        webbrowser.open("https://www.google.com/search?q=farm%C3%A1cia+pr%C3%B3xima")
        return "abrindo uma busca por farmácia próxima."

    if "wikipedia" in comando or "pesquisar" in comando or "pesquise" in comando:
        termo = limpar_termo_wikipedia(comando)

        if not termo:
            return "me diga o que você quer pesquisar na wikipedia."

        try:
            return wikipedia.summary(termo, sentences=2, auto_suggest=False)
        except wikipedia.exceptions.DisambiguationError as erro:
            opcoes = ", ".join(erro.options[:3])
            return f"encontrei mais de um resultado. tenta ser mais específico. opções: {opcoes}."
        except wikipedia.exceptions.PageError:
            url = f"https://pt.wikipedia.org/wiki/Special:Search?search={quote_plus(termo)}"
            webbrowser.open(url)
            return "não encontrei um resumo direto, então abri a busca na wikipedia."
        except Exception:
            return "não consegui pesquisar na wikipedia agora."

    if comando in ["sair", "encerrar", "parar"]:
        return "certo, encerrando o assistente."

    return "ainda não sei responder esse comando."

## pesquisa wikipedia

exemplo de comando pesquisando um assunto na wikipedia.

In [ ]:
resposta = responder_comando("pesquisar python na wikipedia")
speak(resposta)

## abertura de links

o `webbrowser` tenta abrir links no navegador. no colab, dependendo do ambiente, a abertura pode não aparecer como em um computador local.

In [ ]:
resposta = responder_comando("abrir youtube")
speak(resposta)

## exemplo de comando para farmácia próxima

In [ ]:
resposta = responder_comando("farmácia próxima")
speak(resposta)

## teste livre

digite um comando para testar a resposta.

In [ ]:
comando = input("digite um comando: ")
resposta = responder_comando(comando)
speak(resposta)

## conclusão

este notebook mostra a parte principal do assistente funcionando por texto, com geração de áudio e comandos simples.

para testar speech to text com microfone, o ideal é rodar o projeto localmente usando `python src/assistant.py --mode voice`.